In [1]:
# --- Run ↔ Time conversions for M1 & M2 (1 run = 1 hour) ---
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import math

TZ = ZoneInfo("America/New_York")
RUN_DT = datetime(2025, 7, 28, 8, 0, tzinfo=TZ)  # anchor time

ANCHOR = {
    "M1": {"run": 24531, "t": RUN_DT},
    "M2": {"run": 396,   "t": RUN_DT},
}
RUN_LEN = timedelta(hours=1)

def _as_dt(x):
    """Accept datetime or 'YYYY-MM-DD HH:MM' (local NY). Make timezone-aware."""
    if isinstance(x, datetime):
        return x if x.tzinfo else x.replace(tzinfo=TZ)
    return datetime.fromisoformat(x).replace(tzinfo=TZ) if "T" not in x else datetime.fromisoformat(x)

def run_to_time(det, run_number):
    """Exact timestamp for a (possibly float) run_number."""
    a = ANCHOR[det]
    return a["t"] + (run_number - a["run"]) * RUN_LEN

def time_to_run(det, when, rounding="nearest"):
    """
    Convert time -> run number.
    rounding in {"nearest","floor","ceil","none"}; 'none' returns float.
    """
    a = ANCHOR[det]
    t = _as_dt(when)
    run_float = a["run"] + (t - a["t"]) / RUN_LEN
    if rounding == "none":
        return run_float
    if rounding == "floor":
        return math.floor(run_float)
    if rounding == "ceil":
        return math.ceil(run_float)
    return int(round(run_float))  # nearest

# --- examples ---
# M1: run -> time
t_m1 = run_to_time("M1", 24531)              # -> 2025-07-28 08:00 EDT
t_m1_next = run_to_time("M1", 24532)         # -> 2025-07-28 09:00 EDT

# M2: time -> run
r_m2_at_anchor = time_to_run("M2", "2025-07-28 08:00", rounding="nearest")  # -> 396
r_m2_float = time_to_run("M2", "2025-07-28 12:30", rounding="none")         # -> 400.5
r_m2_floor = time_to_run("M2", "2025-07-28 12:30", rounding="floor")        # -> 400

print("M1 24531 ->", t_m1)
print("M1 24532 ->", t_m1_next)
print("M2 2025-07-28 08:00 -> run", r_m2_at_anchor)
print("M2 2025-07-28 12:30 -> run(float)", r_m2_float, " floor:", r_m2_floor)


M1 24531 -> 2025-07-28 08:00:00-04:00
M1 24532 -> 2025-07-28 09:00:00-04:00
M2 2025-07-28 08:00 -> run 396
M2 2025-07-28 12:30 -> run(float) 400.5  floor: 400


In [2]:
time_m1 = "2025-06-01 08:00"
run_m1 = time_to_run("M1", time_m1, rounding="nearest")
print(f"M1 run at {time_m1} is approximately run {run_m1}")

M1 run at 2025-06-01 08:00 is approximately run 23163


In [2]:
import re
from pathlib import Path

# --- Configuration ---
# 1. Set the path to your data folder
folder_path = Path("/raid1/genli/Data_D2O/M1_data/BRN") 

# 2. Set the run number range (inclusive)
run_min = 25306
run_max = 26233

# 3. Define the file name format
# This regex will capture the number (d+) between "run" and "_processed_v5.root"
file_regex = re.compile(r"run(\d+)_processed_v5\.root")

# This glob pattern helps find candidate files quickly
glob_pattern = "run*_processed_v5.root"

# --- Analysis ---
file_count = 0
found_runs = []

print(f"Scanning folder: {folder_path}")
print(f"Looking for runs in range: [{run_min}, {run_max}]")
print(f"Matching pattern: {glob_pattern}\n")

if not folder_path.is_dir():
    print(f"--- ERROR ---")
    print(f"Folder not found. Please check the 'folder_path' variable.")
else:
    # Iterate over files matching the general glob pattern
    for file in folder_path.glob(glob_pattern):
        
        # Check if the filename matches the specific regex
        match = file_regex.match(file.name)
        
        if match:
            try:
                # Extract the captured group (the digits) and convert to int
                run_num = int(match.group(1))
                
                # Check if the run number is in the specified range
                if run_min <= run_num <= run_max:
                    file_count += 1
                    found_runs.append(run_num)
                    # print(f"Found: {file.name}") # Uncomment for a full list
                    
            except ValueError:
                # This would happen if the regex matched something not an integer
                print(f"Warning: Could not parse run number from {file.name}")

    # --- Results ---
    print(f"\n--- Report ---")
    print(f"Total files found in range: {file_count}")

Scanning folder: /raid1/genli/Data_D2O/M1_data/BRN
Looking for runs in range: [25306, 26233]
Matching pattern: run*_processed_v5.root


--- Report ---
Total files found in range: 928
